# NER - Nhận diện thực thể có tên (Named Entity Recognition)
## Nhãn: TIME, TASK, LOCATION, PARTNER

# Setup: Clone hoặc pull repo từ GitHub

In [ ]:
import os

REPO_URL = "https://github.com/CongThuan02/su_ly_ngon_ngu_tu_nhien.git"
REPO_DIR = "/content/su_ly_ngon_ngu_tu_nhien"

if os.path.exists(REPO_DIR):
    print("Repo đã tồn tại, đang pull code mới nhất...")
    os.system(f"git -C {REPO_DIR} pull origin main")
else:
    print("Đang clone repo...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Thư mục hiện tại: {os.getcwd()}")


## Bước 1: Cài đặt thư viện

In [ ]:
!pip install torch transformers seqeval numpy fastapi uvicorn pyngrok nest-asyncio -q

## Bước 2: Sinh dữ liệu tổng hợp

In [ ]:
import json
import random

TASKS = [
    # Không dấu — công việc / học tập
    "hop du an", "bao cao tien do", "demo san pham", "review hop dong",
    "gap khach hang", "cuoc hop", "thuyet trinh", "ky hop dong",
    "hop trien khai", "an toi", "an trua", "an sang", "hop bao",
    "dao tao nhan vien", "phong van ung vien", "hop giao ban", "hop chien luoc",
    "hoc xu ly ngon ngu tu nhien", "hoc machine learning", "hoc lap trinh",
    "on tap", "lam bai tap", "nghien cuu", "doc tai lieu",
    "hoc toan", "hoc lich su", "hoc vat ly", "hoc hoa hoc",
    "lich hoc", "lich hop", "buoi hoc", "buoi hop",
    "hoc mon con lai", "hoc bai", "hoc nhom", "kiem tra",
    # Không dấu — sinh hoạt / sức khoẻ
    "uong thuoc", "tap the duc", "chay bo", "yoga", "kiem tra suc khoe",
    "kham benh", "uong nuoc", "ngu som", "do huyet ap", "lay thuoc",
    "xet nghiem mau", "tai kham", "uong vitamin",
    # Có dấu — công việc / học tập
    "họp dự án", "báo cáo tiến độ", "demo sản phẩm", "review hợp đồng",
    "gặp khách hàng", "cuộc họp", "thuyết trình", "ký hợp đồng",
    "họp triển khai", "ăn tối", "ăn trưa", "ăn sáng", "họp báo",
    "đào tạo nhân viên", "phỏng vấn ứng viên", "họp giao ban", "họp chiến lược",
    "học xử lý ngôn ngữ tự nhiên", "học machine learning", "học lập trình",
    "ôn tập", "làm bài tập", "nghiên cứu", "đọc tài liệu",
    "học toán", "học lịch sử", "học vật lý", "học hóa học",
    "lịch học", "lịch họp", "buổi học", "buổi họp",
    "học môn còn lại", "học bài", "học nhóm", "kiểm tra",
    # Có dấu — sinh hoạt / sức khoẻ
    "uống thuốc", "tập thể dục", "chạy bộ", "yoga", "kiểm tra sức khoẻ",
    "khám bệnh", "uống nước", "ngủ sớm", "đo huyết áp", "lấy thuốc",
    "xét nghiệm máu", "tái khám", "uống vitamin",
]

LOCATIONS = [
    # Không dấu
    "phong hop tang 3", "phong a", "van phong quan 1", "phong meeting 2",
    "nha hang song viet", "phong du an tang 5", "tru so chinh", "phong hop lon",
    "hoi truong A", "phong giam doc", "co so 2", "khach san sofitel",
    "van phong ha noi", "phong hop nho", "phong vip tang 10", "phong tu van",
    # Có dấu
    "phòng họp tầng 3", "phòng A", "văn phòng quận 1", "phòng meeting 2",
    "nhà hàng sông việt", "phòng dự án tầng 5", "trụ sở chính", "phòng họp lớn",
    "hội trường A", "phòng giám đốc", "cơ sở 2", "khách sạn sofitel",
    "văn phòng hà nội", "phòng họp nhỏ", "phòng vip tầng 10", "phòng tư vấn",
    # Online
    "teams", "zoom", "google meet", "skype", "webex",
    "microsoft teams", "google classroom", "zalo", "discord",
]

HO = ["nguyễn", "trần", "lê", "phạm", "hoàng", "huỳnh", "phan", "vũ", "đặng", "bùi",
      "nguyen", "tran", "le", "pham", "hoang", "huynh", "phan", "vu", "dang", "bui"]
TEN = ["an", "bình", "chi", "dũng", "em", "phong", "giang", "hoa", "hùng", "lan",
       "lộc", "mai", "nam", "oanh", "phúc", "quân", "sơn", "thảo", "uyên", "vinh", "dung",
       "an", "binh", "chi", "dung", "em", "phong", "giang", "hoa", "hung", "lan",
       "loc", "mai", "nam", "oanh", "phuc", "quan", "son", "thao", "uyen", "vinh"]

DANH_XUNG_CO_DAU    = ["thầy", "cô", "anh", "chị", "bạn", "ông", "bà", "em", "giáo sư", "tiến sĩ"]
DANH_XUNG_KHONG_DAU = ["thay", "co", "anh", "chi", "ban", "ong", "ba", "em", "giao su", "tien si"]

ORGS = [
    "g-solutions", "cong ty an binh", "seatech", "blue ocean", "sunrise holdings",
    "acme", "tap doan minh phat", "viettel", "fpt software", "cong ty abc",
    "green tech", "smart solutions", "cong ty alpha", "cong ty beta group",
    "công ty an bình", "tập đoàn minh phát", "viettel", "fpt software",
    "công ty abc", "green tech", "smart solutions",
]

TIME_LIST = [
    # Không dấu — dạng "Xh" / "Xh30"
    "sang mai luc {h}h{m}", "chieu mai luc {h}h{m}", "toi nay luc {h}h{m}",
    "sang thu {d} luc {h}h{m}", "ngay {dd}/{mm} luc {h}h{m}",
    "thu {d} luc {h}h{m}", "mai luc {h}h{m}",
    "sang thu {d}", "thu {d} tuan sau luc {h}h{m}", "ngay {dd}/{mm}",
    "{h}h", "{h}h{m}", "luc {h}h", "luc {h}h{m}",
    "hom nay luc {h}h{m}", "hom nay", "sang nay", "chieu nay", "toi nay",
    "{h}h sang", "{h}h chieu", "{h}h toi",
    # Không dấu — dạng "HH:MM"
    "{h}:{m2}", "luc {h}:{m2}", "vao luc {h}:{m2}",
    # Không dấu — chỉ buổi + thứ (không giờ)
    "toi thu {d}", "sang thu {d} tuan sau", "toi thu {d} tuan sau",
    "chieu thu {d} tuan sau",
    # Có dấu — dạng "Xh" / "Xh30"
    "sáng mai lúc {h}h{m}", "chiều mai lúc {h}h{m}", "tối nay lúc {h}h{m}",
    "sáng thứ {d} lúc {h}h{m}", "ngày {dd}/{mm} lúc {h}h{m}",
    "thứ {d} lúc {h}h{m}", "mai lúc {h}h{m}",
    "sáng thứ {d}", "thứ {d} tuần sau lúc {h}h{m}", "ngày {dd}/{mm}",
    "{h}h{m}", "lúc {h}h", "lúc {h}h{m}",
    "hôm nay lúc {h}h{m}", "hôm nay", "sáng nay", "chiều nay", "tối nay",
    "{h}h sáng", "{h}h chiều", "{h}h tối",
    # Có dấu — dạng "HH:MM"
    "{h}:{m2}", "lúc {h}:{m2}", "vào lúc {h}:{m2}",
    # Có dấu — chỉ buổi + thứ (không giờ)
    "tối thứ {d}", "sáng thứ {d} tuần sau", "tối thứ {d} tuần sau",
    "chiều thứ {d} tuần sau",
]

MULTI_TIME_TEMPLATES = [
    "{h1}h và {h2}h",
    "lúc {h1}h và {h2}h",
    "{h1} giờ và {h2} giờ",
    "vào {h1}h và {h2}h",
    "{h1}:{m1} và {h2}:{m2}",
    "{h1}h{m1} và {h2}h{m2}",
]

RECUR_SUFFIX_CO_DAU    = ["hàng ngày", "mỗi ngày", "hằng ngày"]
RECUR_SUFFIX_KHONG_DAU = ["hang ngay", "moi ngay", "hang ngay"]

LOC_PREPS  = ["tại", "ở", "trên", "tai", "o", "tren"]
PART_PREPS = ["với", "cùng", "voi", "cung"]

BAO_VERBS_CO_DAU    = ["bảo", "nhắn", "nói", "thông báo", "nhắc", "báo"]
BAO_VERBS_KHONG_DAU = ["bao", "nhan", "noi", "thong bao", "nhac", "bao"]

GROUPS_CO_DAU    = ["cả lớp", "nhóm", "team", "lớp"]
GROUPS_KHONG_DAU = ["ca lop", "nhom", "team", "lop"]

NHAC_VERBS_CO_DAU    = ["nhắc", "nhớ nhắc", "đặt nhắc"]
NHAC_VERBS_KHONG_DAU = ["nhac", "nho nhac", "dat nhac"]

NOISE_VERBS_CO_DAU    = ["diễn ra", "tổ chức", "được tổ chức", "sẽ diễn ra",
                          "có lịch", "có cuộc", "bắt đầu", "khai mạc"]
NOISE_VERBS_KHONG_DAU = ["dien ra", "to chuc", "duoc to chuc", "se dien ra",
                          "co lich", "co cuoc", "bat dau", "khai mac"]

# Mẫu chủ ngữ nhóm — "lớp mình sẽ X"
GROUP_SUBJECTS_CO_DAU    = ["lớp mình", "chúng mình", "tụi mình", "cả lớp", "nhóm mình"]
GROUP_SUBJECTS_KHONG_DAU = ["lop minh", "chung minh", "tui minh", "ca lop", "nhom minh"]

def random_time():
    h  = random.randint(7, 21)
    m  = random.choice(["", "00", "30", "15", "45"])
    m2 = random.choice(["00", "30", "15", "45"])
    d  = random.randint(2, 7)
    dd = random.randint(1, 28)
    mm = random.randint(1, 12)
    return random.choice(TIME_LIST).format(h=h, m=m, m2=m2, d=d, dd=dd, mm=mm)

def random_partner():
    if random.random() < 0.6:
        if random.random() < 0.5:
            dx  = random.choice(DANH_XUNG_CO_DAU)
            ho  = random.choice([h for h in HO  if any(ord(c) > 127 for c in h)])
            ten = random.choice([t for t in TEN if any(ord(c) > 127 for c in t)])
        else:
            dx  = random.choice(DANH_XUNG_KHONG_DAU)
            ho  = random.choice([h for h in HO  if all(ord(c) <= 127 for c in h)])
            ten = random.choice([t for t in TEN if all(ord(c) <= 127 for c in t)])
        use_ho = random.random() < 0.5
        return f"{dx} {ho} {ten}" if use_ho else f"{dx} {ten}"
    else:
        return random.choice(ORGS)

def is_accented(s):
    return any(ord(c) > 127 for c in s)

def find_span(text, val):
    idx = text.find(val)
    return (idx, idx + len(val)) if idx != -1 else (None, None)

def build_sample():
    task = random.choice(TASKS); location = random.choice(LOCATIONS)
    partner = random_partner(); time_str = random_time()
    loc_prep = random.choice(LOC_PREPS); part_prep = random.choice(PART_PREPS)
    include_partner = random.random() > 0.2
    blocks = [time_str, task, f"{loc_prep} {location}"]
    if include_partner: blocks.append(f"{part_prep} {partner}")
    random.shuffle(blocks)
    text = " ".join(blocks)
    entities = []
    for val, label in [(time_str,"TIME"),(task,"TASK"),(location,"LOCATION"),(partner,"PARTNER")]:
        if label == "PARTNER" and not include_partner: continue
        s, e = find_span(text, val)
        if s is not None: entities.append({"start": s, "end": e, "label": label})
    return {"text": text, "entities": entities} if entities else None

def build_sample_noise_verb():
    task = random.choice(TASKS); location = random.choice(LOCATIONS)
    time_str = random_time(); loc_prep = random.choice(LOC_PREPS)
    accented = is_accented(task)
    noise = random.choice(NOISE_VERBS_CO_DAU if accented else NOISE_VERBS_KHONG_DAU)
    time_prep = random.choice(["lúc","vào lúc","vào",""] if accented else ["luc","vao luc","vao",""])
    if random.random() < 0.5:
        text = f"{task} {noise} {time_prep} {time_str} {loc_prep} {location}".strip()
    else:
        text = f"{task} {time_prep} {time_str} {noise} {loc_prep} {location}".strip()
    import re as _re; text = _re.sub(r"\s+", " ", text).strip()
    if random.random() < 0.5: text = text[0].upper() + text[1:]
    entities = []
    for val, label in [(time_str,"TIME"),(task,"TASK"),(location,"LOCATION")]:
        lt = text.lower(); lv = val.lower(); idx = lt.find(lv)
        if idx != -1: entities.append({"start": idx, "end": idx+len(val), "label": label})
    return {"text": text, "entities": entities} if entities else None

def build_sample_bao():
    ps = random_partner(); time_str = random_time()
    task = random.choice(TASKS); location = random.choice(LOCATIONS)
    verb = random.choice(BAO_VERBS_CO_DAU if is_accented(ps) else BAO_VERBS_KHONG_DAU)
    loc_prep = random.choice(LOC_PREPS)
    if random.random() < 0.5:
        grp = random.choice(GROUPS_CO_DAU if is_accented(ps) else GROUPS_KHONG_DAU)
        conj = "và" if is_accented(ps) else "va"
        pp = "với" if is_accented(ps) else "voi"
        partner_phrase = f"{pp} {ps} {conj} {grp}"
    else:
        pp = "với" if is_accented(ps) else "voi"
        partner_phrase = f"{pp} {ps}"
    parts = [time_str, task, partner_phrase, f"{loc_prep} {location}"]
    random.shuffle(parts)
    text = f"{ps} {verb} {' '.join(parts)}"
    entities = []
    for val, label in [(ps,"PARTNER"),(time_str,"TIME"),(task,"TASK"),(location,"LOCATION")]:
        s, e = find_span(text, val)
        if s is not None: entities.append({"start":s,"end":e,"label":label})
    return {"text": text, "entities": entities} if entities else None

def build_sample_recurring():
    task = random.choice(TASKS)
    h1 = random.randint(6,12); h2 = random.randint(13,21)
    m = random.choice(["","00","30"]); m2 = random.choice(["00","30"])
    accented = is_accented(task)
    nhac = random.choice(NHAC_VERBS_CO_DAU if accented else NHAC_VERBS_KHONG_DAU)
    recur = random.choice(RECUR_SUFFIX_CO_DAU if accented else RECUR_SUFFIX_KHONG_DAU)
    tmpl = random.choice(MULTI_TIME_TEMPLATES)
    time_str = tmpl.format(h1=h1,h2=h2,m1=m,m2=m2)
    prep = random.choice(["vào","lúc",""] if accented else ["vao","luc",""])
    text = f"{nhac} {task} {prep} {time_str} {recur}".strip()
    import re as _re; text = _re.sub(r"\s+", " ", text)
    entities = []
    s, e = find_span(text, task)
    if s is not None: entities.append({"start":s,"end":e,"label":"TASK"})
    for h in [h1, h2]:
        for p in [f"{h}h", f"{h} giờ", f"{h} gio", f"{h}:"]:
            si = text.find(p)
            if si != -1:
                ei = si + len(p.rstrip(":"))
                if not any(e["start"]==si for e in entities if e["label"]=="TIME"):
                    entities.append({"start":si,"end":ei,"label":"TIME"})
                break
    return {"text": text, "entities": entities} if entities else None

# ── Pattern mới: "GROUP_SUBJECT sẽ TASK lúc SESSION_TIME với PARTNER" ─────────
def build_sample_group_subject():
    """
    Mô phỏng câu dạng:
      "tối thứ 3 tuần sau lớp mình học môn còn lại của cô Dung"
      "sang thu 4 tuan sau nhom minh hop du an voi anh nam"
    """
    task     = random.choice(TASKS)
    partner  = random_partner()
    time_str = random_time()
    accented = is_accented(task)

    subj = random.choice(GROUP_SUBJECTS_CO_DAU if accented else GROUP_SUBJECTS_KHONG_DAU)
    pp   = "với" if accented else "voi"

    include_partner = random.random() > 0.3
    if include_partner:
        text = f"{time_str} {subj} {task} {pp} {partner}"
    else:
        text = f"{time_str} {subj} {task}"

    # Viết hoa chữ đầu (40%)
    if random.random() < 0.4:
        text = text[0].upper() + text[1:]

    entities = []
    for val, label in [(time_str,"TIME"),(task,"TASK"),(partner,"PARTNER")]:
        if label=="PARTNER" and not include_partner: continue
        s, e = find_span(text, val)
        if s is not None: entities.append({"start":s,"end":e,"label":label})
    return {"text": text, "entities": entities} if entities else None

def generate(n=3000):
    samples, seen = [], set()
    attempts = 0
    while len(samples) < n and attempts < n * 10:
        attempts += 1
        r = random.random()
        if r < 0.10:   s = build_sample_bao()
        elif r < 0.18: s = build_sample_recurring()
        elif r < 0.30: s = build_sample_noise_verb()
        elif r < 0.42: s = build_sample_group_subject()   # ← mới
        else:          s = build_sample()
        if s and s["text"] not in seen:
            seen.add(s["text"]); samples.append(s)
    return samples

random.seed(42)

existing = []
with open("data/train.jsonl") as f:
    for line in f:
        line = line.strip()
        if line: existing.append(json.loads(line))

generated = generate(3000)
all_data = existing + generated
random.shuffle(all_data)

split = int(len(all_data) * 0.8)
train_data = all_data[:split]
test_data  = all_data[split:]

with open("data/train_full.jsonl", "w") as f:
    for s in train_data: f.write(json.dumps(s, ensure_ascii=False) + "\n")
with open("data/test.jsonl", "w") as f:
    for s in test_data:  f.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"Train: {len(train_data)} mẫu")
print(f"Test:  {len(test_data)} mẫu")

print("\nVí dụ pattern 'group subject / tuần sau':")
ex = [s for s in train_data if any(
    v in s["text"].lower() for v in ["lớp mình","tụi mình","tuần sau","tuan sau"]
)][:5]
for s in ex:
    print(f"\n  text: {s['text']}")
    for e in sorted(s["entities"], key=lambda x: x["start"]):
        print(f"  [{e['label']}] '{s['text'][e['start']:e['end']]}'")


## Bước 3: Định nghĩa nhãn và Dataset

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "vinai/phobert-base"
MAX_LEN    = 128

LABELS   = ["O", "B-TIME", "I-TIME", "B-TASK", "I-TASK", "B-LOCATION", "I-LOCATION", "B-PARTNER", "I-PARTNER"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

def char_to_bio(text, entities):
    tokens = text.split()
    char_labels = ["O"] * len(text)
    for ent in entities:
        for i in range(ent["start"], ent["end"]):
            if i < len(char_labels):
                char_labels[i] = ("B-" if i == ent["start"] else "I-") + ent["label"]
    word_labels, pos = [], 0
    for token in tokens:
        word_labels.append(char_labels[pos] if pos < len(char_labels) else "O")
        pos += len(token) + 1
    return tokens, word_labels

def get_word_ids(tokens, tokenizer, max_length):
    """
    Tính word_ids thủ công cho non-fast tokenizer (PhoBERT / SentencePiece).
    Trả về list độ dài max_length:
      - None  → token đặc biệt ([CLS], [SEP], [PAD])
      - int   → chỉ số word gốc trong `tokens`
    """
    # [CLS]
    word_ids = [None]
    for word_idx, word in enumerate(tokens):
        sub = tokenizer.tokenize(word)
        if not sub:                        # từ không tokenize được → unk
            sub = [tokenizer.unk_token]
        word_ids.extend([word_idx] * len(sub))
        if len(word_ids) >= max_length - 1:  # chừa chỗ cho [SEP]
            break
    # [SEP]
    word_ids.append(None)
    # [PAD]
    word_ids += [None] * (max_length - len(word_ids))
    return word_ids[:max_length]

class NERDataset(Dataset):
    def __init__(self, samples, tokenizer):
        self.data = []
        for sample in samples:
            tokens, word_labels = char_to_bio(sample["text"], sample["entities"])
            enc = tokenizer(
                tokens,
                is_split_into_words=True,
                truncation=True,
                max_length=MAX_LEN,
                padding="max_length",
                return_tensors="pt",
            )
            word_ids = get_word_ids(tokens, tokenizer, MAX_LEN)

            label_ids, prev = [], None
            for wid in word_ids:
                if wid is None:
                    label_ids.append(-100)
                elif wid != prev:
                    label_ids.append(LABEL2ID.get(word_labels[wid], 0))
                else:
                    tag = word_labels[wid]
                    if tag.startswith("B-"):
                        tag = "I-" + tag[2:]
                    label_ids.append(LABEL2ID.get(tag, 0))
                prev = wid

            self.data.append({
                "input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels":         torch.tensor(label_ids, dtype=torch.long),
            })

    def __len__(self):         return len(self.data)
    def __getitem__(self, i):  return self.data[i]

print("Đang tải tokenizer PhoBERT...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
train_dataset = NERDataset(train_data, tokenizer)
test_dataset  = NERDataset(test_data,  tokenizer)
print(f"Dataset sẵn sàng — Train: {len(train_dataset)}, Test: {len(test_dataset)}")


## Bước 4: Huấn luyện mô hình

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import classification_report, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_labels, pred_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        tr, pr = [], []
        for p, l in zip(pred_seq, label_seq):
            if l == -100: continue
            tr.append(ID2LABEL[l]); pr.append(ID2LABEL[p])
        true_labels.append(tr); pred_labels.append(pr)
    return {"f1": f1_score(true_labels, pred_labels)}

print("Đang tải model PhoBERT...")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

args = TrainingArguments(
    output_dir="model_output",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Bắt đầu huấn luyện...")
trainer.train()


## Bước 5: Đánh giá & lưu model

In [ ]:
results = trainer.evaluate()
print(f"Test F1: {results['eval_f1']:.4f}")

pred_out = trainer.predict(test_dataset)
preds = np.argmax(pred_out.predictions, axis=-1)
labels = pred_out.label_ids
true_labels, pred_labels = [], []
for pred_seq, label_seq in zip(preds, labels):
    tr, pr = [], []
    for p, l in zip(pred_seq, label_seq):
        if l == -100: continue
        tr.append(ID2LABEL[l]); pr.append(ID2LABEL[p])
    true_labels.append(tr); pred_labels.append(pr)

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels))

trainer.save_model("model_output")
tokenizer.save_pretrained("model_output")
print("Model đã lưu vào model_output/")


## Bước 6: Dự đoán văn bản mới

In [ ]:
def predict(text: str) -> dict:
    tokens = text.split()
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )
    with torch.no_grad():
        outputs = model(**enc)

    preds    = torch.argmax(outputs.logits[0], dim=-1).tolist()
    word_ids = get_word_ids(tokens, tokenizer, len(preds))

    word_preds, prev = [], None
    for pid, wid in zip(preds, word_ids):
        if wid is None or wid == prev:
            prev = wid
            continue
        word_preds.append((tokens[wid], ID2LABEL[pid]))
        prev = wid

    # Gom BIO → danh sách entity thô
    raw_entities, current = [], None
    for word, label in word_preds:
        if label.startswith("B-"):
            if current: raw_entities.append(current)
            current = {"text": word, "label": label[2:]}
        elif label.startswith("I-") and current:
            current["text"] += " " + word
        else:
            if current: raw_entities.append(current)
            current = None
    if current: raw_entities.append(current)

    # Gom thành dict — PARTNER là list (có thể nhiều người/tổ chức)
    entities_dict = {}
    for ent in raw_entities:
        lbl = ent["label"]
        if lbl == "PARTNER":
            entities_dict.setdefault("PARTNER", []).append(ent["text"])
        else:
            if lbl not in entities_dict:
                entities_dict[lbl] = ent["text"]

    return {"text": text, "entities": [entities_dict]}

# Test thử — bao gồm các input đã gặp
test_cases = [
    "thầy lộc bảo 20h học với thầy và lớp ở teams",
    "anh nam nhắn sáng mai 9h họp dự án với nhóm tại phòng họp tầng 3",
    "hôm nay có lịch học lúc 8h tối với thầy lộc trên teams",
    "sáng mai lúc 9h họp dự án với công ty FPT tại phòng họp tầng 3",
    "nhắc uống thuốc vào 12 giờ và 20 giờ hàng ngày",
]
for text in test_cases:
    result = predict(text)
    print(f"Text: {text}")
    for k, v in result["entities"][0].items():
        print(f"  [{k}] {v}")
    print()


## Bước 7: Giao diện nhập văn bản

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

text_input = widgets.Textarea(
    placeholder="Nhập đoạn văn bản tiếng Việt vào đây...",
    layout=widgets.Layout(width="100%", height="80px"),
)
btn = widgets.Button(description="Phân tích", button_style="primary",
                     layout=widgets.Layout(width="120px"))
output = widgets.Output()

COLORS = {"TIME": "#9C27B0", "TASK": "#4CAF50", "LOCATION": "#2196F3", "PARTNER": "#FF9800"}

def on_click(b):
    with output:
        clear_output()
        text = text_input.value.strip()
        if not text:
            print("Vui lòng nhập văn bản!"); return
        result = predict(text)
        entities = result["entities"][0]
        if not entities:
            print("Không tìm thấy thực thể nào."); return

        highlighted = text
        for label, val in entities.items():
            color = COLORS.get(label, "#9E9E9E")
            highlighted = highlighted.replace(val,
                f'<mark style="background:{color};color:white;padding:2px 6px;'
                f'border-radius:4px;margin:0 2px;">{val} '
                f'<sup style="font-size:0.7em">{label}</sup></mark>')
        display(HTML(f"<div style='font-size:16px;line-height:2'>{highlighted}</div>"))

        rows = "".join(
            f"<tr><td style='padding:6px 12px'><b style='color:{COLORS.get(k,'#000')}'>{k}</b></td>"
            f"<td style='padding:6px 12px'>{v}</td></tr>"
            for k, v in entities.items()
        )
        display(HTML(
            f"<table border='1' cellspacing='0' style='margin-top:12px;border-collapse:collapse'>"
            f"<tr style='background:#f0f0f0'><th style='padding:6px 12px'>Nhãn</th>"
            f"<th style='padding:6px 12px'>Thực thể</th></tr>{rows}</table>"
        ))

btn.on_click(on_click)
display(text_input, btn, output)


## Bước 8: Chạy API Server (cho Flutter gọi)

In [ ]:
# Cài FastAPI, uvicorn, pyngrok
!pip install fastapi uvicorn pyngrok nest-asyncio -q

In [ ]:
# Điền ngrok token của bạn vào đây
# Đăng ký miễn phí tại: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "2QeB0CzXXsDXmRl2JsO2SVDaXyC_2rzPsuxEUEFAnGg4DjAfD"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class TextInput(BaseModel):
    text: str

@app.get("/")
def root():
    return {"status": "NER API đang chạy"}

@app.post("/predict")
def api_predict(input: TextInput):
    return predict(input.text)

# Tạo URL public qua ngrok
public_url = ngrok.connect(8000)
print("=" * 50)
print(f"API URL: {public_url}")
print(f"Flutter gọi tới: {public_url}/predict")
print("=" * 50)

# Chạy server
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="error")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run)
thread.start()

## Flutter - Code mẫu gọi API

## Dán URL ngrok vào Flutter như sau:

```dart
// pubspec.yaml: thêm http: ^1.0.0

import 'dart:convert';
import 'package:http/http.dart' as http;

// Thay URL ngrok in ra ở trên vào đây
const String API_URL = "https://xxxx-xxxx.ngrok-free.app/predict";

Future<List<dynamic>> predictNER(String text) async {
  final response = await http.post(
    Uri.parse(API_URL),
    headers: {
      'Content-Type': 'application/json',
      'ngrok-skip-browser-warning': '1',  // bỏ qua trang cảnh báo của ngrok
    },
    body: jsonEncode({'text': text}),
  );

  if (response.statusCode == 200) {
    final data = jsonDecode(response.body);
    return data['entities'];  // [{text: "20h", label: "TIME"}, ...]
  } else {
    throw Exception('Lỗi API: ${response.statusCode}');
  }
}
```